In [1]:
# Import the necessary libraries and modules.

import numpy as np
import sklearn
import pandas as pd

from matplotlib import pyplot as plt

import seaborn as sns

from sklearn.metrics import mean_squared_error, mean_absolute_error

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Lasso

from sklearn.feature_selection import SelectKBest, f_regression, SelectFromModel

from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import TimeSeriesSplit

from sklearn.pipeline import Pipeline

from tabulate import tabulate

In [2]:
# Read the time series data for a specific building (Building_1) from a CSV file using pandas' read_csv function.
# The file likely contains information such as electricity consumption, production, or related metrics for Building 1.
building1_data = pd.read_csv("../Dataset/Building_1.csv")

# Print the electricity data to the console for inspection.
print(building1_data)

     month  hour  day_type  daylight_savings_status  \
0        6     1         5                        0   
1        6     2         5                        0   
2        6     3         5                        0   
3        6     4         5                        0   
4        6     5         5                        0   
..     ...   ...       ...                      ...   
715      6    20         6                        0   
716      6    21         6                        0   
717      6    22         6                        0   
718      6    23         6                        0   
719      6    24         6                        0   

     indoor_dry_bulb_temperature  average_unmet_cooling_setpoint_difference  \
0                      23.098652                                  -0.123570   
1                      22.234743                                   0.012520   
2                      22.223060                                   0.000838   
3                      

In [3]:
# Read the time series data for Building 2 from a CSV file using pandas' read_csv function.
building2_data = pd.read_csv("../Dataset/Building_2.csv")

# Remove unwanted columns from the dataset using pandas' drop function.
# The columns 'hour', 'month', 'day_type', 'daylight_savings_status', and 'solar_generation' are dropped, as they contain the same values across all buildings
building2_data = building2_data.drop(columns=['hour', 'month', 'day_type', 'daylight_savings_status', 'solar_generation'])

# Rename the remaining columns to more descriptive names that end with '_2' to indicate they belong to Building 2.
# Each column represents different measurements, such as temperature, humidity, energy demand, and occupancy.
building2_data.columns = ['indoor_dry_bulb_temperature_2',
       'average_unmet_cooling_setpoint_difference_2', 'indoor_relative_humidity_2',
       'non_shiftable_load_2', 'dhw_demand_2', 'cooling_demand_2', 'heating_demand_2',
       'occupant_count_2', 'indoor_dry_bulb_temperature_set_point_2', 'hvac_mode_2']

# Print the processed building data for inspection.
print(building2_data)

     indoor_dry_bulb_temperature_2  \
0                        24.278513   
1                        24.264734   
2                        24.214113   
3                        24.119434   
4                        23.995500   
..                             ...   
715                      24.995880   
716                      24.997467   
717                      24.997482   
718                      25.108414   
719                      25.548622   

     average_unmet_cooling_setpoint_difference_2  indoor_relative_humidity_2  \
0                                      -0.165932                   60.703620   
1                                      -0.179711                   61.140690   
2                                      -0.230331                   61.779377   
3                                      -0.325011                   62.557430   
4                                      -0.448944                   64.078804   
..                                           ...               

In [4]:
# Read the time series data for Building 3 from a CSV file using pandas' read_csv function.
building3_data = pd.read_csv("../Dataset/Building_3.csv")

# Remove unwanted columns from the dataset using pandas' drop function.
# The columns 'hour', 'month', 'day_type', 'daylight_savings_status', and 'solar_generation' are dropped, as they contain the same values across all buildings
building3_data = building3_data.drop(columns=['hour', 'month', 'day_type', 'daylight_savings_status', 'solar_generation'])

# Rename the remaining columns to more descriptive names that end with '_3' to indicate they belong to Building 3.
# Each column represents different measurements, such as temperature, humidity, energy demand, and occupancy.
building3_data.columns = ['indoor_dry_bulb_temperature_3',
       'average_unmet_cooling_setpoint_difference_3', 'indoor_relative_humidity_3',
       'non_shiftable_load_3', 'dhw_demand_3', 'cooling_demand_3', 'heating_demand_3',
       'occupant_count_3', 'indoor_dry_bulb_temperature_set_point_3', 'hvac_mode_3']

# Print the processed building data for inspection.
print(building3_data)


     indoor_dry_bulb_temperature_3  \
0                        24.431562   
1                        24.444384   
2                        24.444445   
3                        24.444439   
4                        24.361916   
..                             ...   
715                      24.444447   
716                      24.444450   
717                      24.444450   
718                      24.444450   
719                      24.444440   

     average_unmet_cooling_setpoint_difference_3  indoor_relative_humidity_3  \
0                                  -1.288156e-02                   57.864120   
1                                  -6.162604e-05                   58.086220   
2                                   9.792859e-07                   58.383118   
3                                  -4.807433e-06                   59.098373   
4                                  -8.252885e-02                   60.374480   
..                                           ...               

In [5]:
# Read the carbon intensity data from a CSV file using pandas' read_csv function.
# The file likely contains data on carbon intensity over time, which may refer to the amount of carbon dioxide emitted per unit of energy.
carbon_data = pd.read_csv("../Dataset/carbon_intensity.csv")

# Print the carbon intensity data to the console for inspection.
print(carbon_data)

     carbon_intensity
0            0.402488
1            0.382625
2            0.369458
3            0.367017
4            0.374040
..                ...
715          0.465811
716          0.470324
717          0.462414
718          0.448648
719          0.428057

[720 rows x 1 columns]


In [6]:
# Read the pricing data from a CSV file named "pricing.csv" using pandas' read_csv function.
# The file is expected to contain pricing information relevant to the CityLearn Challenge 2023.
pricing_data = pd.read_csv("../Dataset/pricing.csv")

# Print the contents of the pricing data to the console for inspection.
print(pricing_data)

     electricity_pricing  electricity_pricing_predicted_6h  \
0                0.02893                           0.02893   
1                0.02893                           0.02915   
2                0.02893                           0.02915   
3                0.02893                           0.02915   
4                0.02893                           0.02915   
..                   ...                               ...   
715              0.02893                           0.02893   
716              0.02893                           0.02893   
717              0.02893                           0.02893   
718              0.02893                           0.02893   
719              0.02893                           0.02893   

     electricity_pricing_predicted_12h  electricity_pricing_predicted_24h  
0                              0.02915                            0.02893  
1                              0.02915                            0.02893  
2                          

In [7]:
# Read the weather data from a CSV file named "weather.csv" using pandas' read_csv function.
# The file is expected to contain weather information relevant to the CityLearn Challenge 2023.
weather_data = pd.read_csv("../Dataset/weather.csv")

# Print the contents of the weather data to the console for inspection.
print(weather_data)

     outdoor_dry_bulb_temperature  outdoor_relative_humidity  \
0                           24.66                      77.56   
1                           24.07                      85.12   
2                           23.90                      89.62   
3                           23.87                      91.88   
4                           23.83                      93.06   
..                            ...                        ...   
715                         31.98                      43.75   
716                         29.92                      51.62   
717                         28.48                      59.12   
718                         27.27                      66.56   
719                         26.26                      73.75   

     diffuse_solar_irradiance  direct_solar_irradiance  \
0                        0.00                     0.00   
1                        0.00                     0.00   
2                        0.00                     0.00   

In [8]:
# Concatenate multiple datasets along the columns (axis=1) to create a single dataset.
# The datasets being combined are:
# - electricity_data: Data related to the building's time series data.
# - carbon_data: Data related to carbon emissions or levels.
# - pricing_data: Data related to pricing (could be electricity or carbon pricing).
# - weather_data: Data related to weather conditions.
# The resulting dataset will contain all the columns from these individual datasets.
df = pd.concat([building1_data, building2_data, building3_data, carbon_data, pricing_data, weather_data], axis=1)

# Print the combined dataset to the console to inspect the data.
print(df)

     month  hour  day_type  daylight_savings_status  \
0        6     1         5                        0   
1        6     2         5                        0   
2        6     3         5                        0   
3        6     4         5                        0   
4        6     5         5                        0   
..     ...   ...       ...                      ...   
715      6    20         6                        0   
716      6    21         6                        0   
717      6    22         6                        0   
718      6    23         6                        0   
719      6    24         6                        0   

     indoor_dry_bulb_temperature  average_unmet_cooling_setpoint_difference  \
0                      23.098652                                  -0.123570   
1                      22.234743                                   0.012520   
2                      22.223060                                   0.000838   
3                      

In [9]:
# Define the target variable to be predicted
target = "carbon_intensity"

# Create lagged features for the target from 2 and 48 hours earlier
df["carbon_intensity_lag_24"] = df[target].shift(24)
df["carbon_intensity_lag_48"] = df[target].shift(48)

# Remove rows containing NaN values introduced by the lag operation
df = df.dropna().reset_index(drop=True)

In [10]:
# Split the dataset into input features and target variable
X = df.drop(target, axis=1)
y = df[target]

In [11]:
# Define the number of observations per day
hour_per_day = 24
# Compute the total number of complete days available in the dataset
days = int(len(df)/hour_per_day)

# Set the first day from which the forecasting evaluation starts
start_day = 22

In [12]:
# Determine the end inde of the inizal training set
training_end = start_day * hour_per_day

# Create the initial training subset for features and target
X_train_init = X.iloc[:training_end]
y_train_init = y.iloc[:training_end]

In [13]:
# Create a time series cross-validation strategy that preserves the chronological order
tscv = TimeSeriesSplit(n_splits=3)

# Define the hyperparameter search space for the model
param_grid = {
    "n_estimators": [50, 100],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2]
}

In [14]:
# Perform hyperparameter tuning using GridSearchCV and tscv with MAE as the evalutation metric
grid = GridSearchCV(estimator=RandomForestRegressor(random_state=42), param_grid=param_grid, cv = tscv, scoring="neg_mean_absolute_error", n_jobs=-1)

# Train all candidate models and identify the best parameter
grid.fit(X_train_init, y_train_init)

# Store the best hyperparameter
best_params = grid.best_params_

print(f"Best params: {best_params}")

Best params: {'max_depth': 10, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 50}


In [15]:
# Initialize the model using the optimal hyperparameter
model = RandomForestRegressor(n_estimators=best_params["n_estimators"], max_depth=best_params["max_depth"], min_samples_split=best_params["min_samples_split"], min_samples_leaf=best_params["min_samples_leaf"], random_state=42, n_jobs=-1)

# Define the feature selection techniques to be evaluated
feature_selection = ["NoFS", "KBest", "Lasso", "RF"]

# Initialize a list to store the evaluation results
results = []

In [16]:
# Evaluate the model using different feature selection methods through a Time Series Cross Validation
for feature in feature_selection:
    # Store the daily error metrics for the current feature selection method
    nmae_list  = []
    nrmse_list = []

    # Iterate through the dataset day by day
    for day in range(start_day, days):
        #Define training and testing windows
        train_end = day * hour_per_day
        test_start = train_end
        test_end = test_start + hour_per_day

        # Create expanding trainig sets and one-day-ahead test sets
        X_train = X.iloc[:train_end]
        y_train = y.iloc[:train_end]

        X_test = X.iloc[test_start:test_end]
        y_test = y.iloc[test_start:test_end]

        # Apply the selected feature selection technique
        if feature == "NoFS":
            X_train_sel = X_train
            X_test_sel = X_test
        elif feature == "KBest":
            selector = SelectKBest(score_func=f_regression, k = 10)

            X_train_sel = selector.fit_transform(X_train, y_train)
            X_test_sel = selector.transform(X_test)

        elif feature == "Lasso":
            selector = SelectFromModel(Lasso(alpha=0.01))

            X_train_sel = selector.fit_transform(X_train, y_train)
            X_test_sel = selector.transform(X_test)

        elif feature == "RF":
            selector = SelectFromModel(RandomForestRegressor(n_estimators=100, random_state=42))

            X_train_sel = selector.fit_transform(X_train, y_train)
            X_test_sel = selector.transform(X_test)
        
        # Train the model and generate predictions
        model.fit(X_train_sel, y_train)

        y_pred = model.predict(X_test_sel)

        # Compute prediction errors
        mae = mean_absolute_error(y_test, y_pred)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))

        # Normalize the error metrics using the target range
        nmae = mae / (y_test.max() - y_test.min())
        nrmse = rmse / (y_test.max() - y_test.min())

        # Store daily results
        nmae_list.append(nmae)
        nrmse_list.append(nrmse)
    
    # Compute the average performance across all forecasting days
    nmae = np.mean(nmae_list)
    nrmse = np.mean(nrmse_list)

    # Save the aggregated results for the current feature selection method
    results.append({"Model": "Random Forest", "FS": feature, "NMAE": nmae, "NRMSE": nrmse})
    

In [17]:
# Display the final performance comparison table for all 
# feature selection methods evaluated with the model.
print(tabulate(results, headers="keys", tablefmt="grid"))

+---------------+-------+----------+----------+
| Model         | FS    |     NMAE |    NRMSE |
+===============+=======+==========+==========+
| Random Forest | NoFS  | 0.217955 | 0.273833 |
+---------------+-------+----------+----------+
| Random Forest | KBest | 0.243372 | 0.296301 |
+---------------+-------+----------+----------+
| Random Forest | Lasso | 0.228439 | 0.287894 |
+---------------+-------+----------+----------+
| Random Forest | RF    | 0.221471 | 0.284037 |
+---------------+-------+----------+----------+
